<div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);padding:32px 28px;border-radius:12px;margin-bottom:8px">
<h1 style="color:#e9c46a;margin:0 0 6px">⚙️ FIFA World Cup 2026</h1>
<h2 style="color:#a8dadc;font-weight:400;margin:0 0 14px">Notebook 2 of 4 — Feature Engineering</h2>
<p style="color:#ccc;line-height:1.6;margin:0 0 16px">
We build 16 features from scratch on 49K+ international matches — ELO ratings with
decay and tournament weighting, rolling form, and head-to-head records.
All features are computed <strong style="color:#e9c46a">leak-free</strong>: at each match we only use data from before it.
</p>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#2a9d8f">⚙️ Features</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#e76f51;margin-left:6px">🔢 16 Features</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#6a4c93;margin-left:6px">💾 Saves features CSV</span>
</div>

---
### Feature groups built here

| Group | Features | Key design decision |
|-------|----------|---------------------|
| ELO strength | `home_elo_pre`, `away_elo_pre`, `elo_diff` | Built from ALL 49K matches, not just WC |
| Recent form | `home/away_form_wr`, `_gf`, `_ga`, `form_wr_diff` | Rolling last-10, pre-match snapshot |
| Competitive form | `home/away_form_comp` | Friendlies excluded |
| Head-to-head | `h2h_home_wr`, `h2h_away_wr` | Full historical win rate |
| Match context | `is_neutral`, `is_wc` | WC matches flagged separately |

> **Prerequisite:** Run `01_eda.ipynb` first (or just have the data files in `data/`)  
> **Output:** `features/historical_features.csv` — loaded by notebook 03


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, deque
import os, warnings
warnings.filterwarnings("ignore")

os.makedirs("features", exist_ok=True)

PALETTE = ["#2a9d8f","#e9c46a","#e76f51","#264653","#a8dadc",
           "#457b9d","#fca311","#6a4c93","#f4a261","#1d3557"]
print("✅ Ready")

## 2. Load Data

In [ ]:
intl = pd.read_csv("data/results.csv")
intl["date"] = pd.to_datetime(intl["date"])
intl = intl.sort_values("date").reset_index(drop=True)

historical = intl[intl["home_score"].notna()].copy()
wc26_fix   = intl[(intl["tournament"]=="FIFA World Cup") &
                  (intl["date"] >= "2026-01-01") &
                  (intl["home_score"].isna())].copy()

print(f"Historical matches : {len(historical):,}")
print(f"WC 2026 fixtures   : {len(wc26_fix)}")
historical.tail(3)

## 3. ELO Rating System

### Design decisions
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| K — WC match | 40 | Most important matches |
| K — Major tournament | 30 | Euros, Copa, AFCON etc |
| K — Qualifier | 20 | Meaningful but lower stakes |
| K — Friendly | 10 | Low signal |
| ELO decay | 0.98/yr | Inactive teams regress to 1500 |
| Home advantage | +50 pts | Non-neutral venues only |
| Goal margin multiplier | up to 2.5× | Bigger wins = bigger shift |

> **Why decay matters:** In the original model, Turkey's ELO was frozen since 2002 (last WC appearance). Decay ensures they regress toward 1500 each year of inactivity — so they're no longer treated as a mystery contender.


In [ ]:
K_WC=40; K_MAJOR=30; K_QUALIFIER=20; K_FRIENDLY=10
ELO_DECAY = 0.98

def k_factor(tournament):
    t = str(tournament).lower()
    if "world cup" in t and "qualif" not in t: return K_WC
    if any(x in t for x in ["copa america","euro ","africa cup","asian cup",
                              "gold cup","nations league","nations cup"]): return K_MAJOR
    if "qualif" in t or "qualifier" in t: return K_QUALIFIER
    if "friendly" in t: return K_FRIENDLY
    return 20

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

elo         = defaultdict(lambda: 1500.0)
last_played = {}
home_elos_h, away_elos_h = [], []

for _, m in historical.iterrows():
    home, away = m["home_team"], m["away_team"]
    hg, ag     = m["home_score"], m["away_score"]

    for team in [home, away]:
        if team in last_played:
            gap = (m["date"] - last_played[team]).days / 365
            if gap > 0.5:
                elo[team] = 1500 + (elo[team] - 1500) * (ELO_DECAY ** gap)

    home_elos_h.append(elo[home])
    away_elos_h.append(elo[away])

    k      = k_factor(m["tournament"])
    margin = abs(hg - ag)
    mult   = min(1.0 + max(0, margin - 1) * 0.2, 2.5)
    score  = 1.0 if hg > ag else (0.0 if hg < ag else 0.5)
    h_adv  = 0 if m.get("neutral", False) else 50
    delta  = k * mult * (score - expected_score(elo[home] + h_adv, elo[away]))
    elo[home] += delta; elo[away] -= delta
    last_played[home] = m["date"]; last_played[away] = m["date"]

historical["home_elo_pre"] = home_elos_h
historical["away_elo_pre"] = away_elos_h
print("✅ ELO computed")

In [ ]:
# Top 20 ELO chart
top20  = sorted(elo.items(), key=lambda x: -x[1])[:20]
elo_df = pd.DataFrame(top20, columns=["Team","ELO"])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(elo_df["Team"][::-1], elo_df["ELO"][::-1], color=PALETTE[0], edgecolor="white")
ax.axvline(1500, color="gray", linestyle="--", alpha=0.5, label="Baseline = 1500")
for bar, val in zip(bars, elo_df["ELO"][::-1]):
    ax.text(val+3, bar.get_y()+bar.get_height()/2, f"{val:.0f}", va="center", fontsize=9)
ax.set_title("Top 20 Teams by Final ELO (all 49K+ matches)", fontsize=13, fontweight="bold")
ax.set_xlabel("ELO Rating"); ax.legend()
plt.tight_layout(); plt.show()

print("\n=== Sanity check — key WC 2026 teams ===")
for t in ["France","Brazil","Argentina","England","Spain","Germany",
          "Portugal","Netherlands","Turkey","Iran","Saudi Arabia","Qatar"]:
    print(f"  {t:>20s}: {elo[t]:.0f}")

## 4. Rolling Form Features (last 10 matches)

In [ ]:
FORM_W    = 10
team_form = defaultdict(lambda: deque(maxlen=FORM_W))
h_wr,h_gf,h_ga,h_cw = [],[],[],[]
a_wr,a_gf,a_ga,a_cw = [],[],[],[]

def form_snap(team):
    fm = team_form[team]
    if len(fm) < 3: return 0.4, 1.2, 1.2, 0.4
    wins  = sum(1 for g,a,c in fm if g > a)
    cwins = sum(1 for g,a,c in fm if g > a and c)
    return (wins/len(fm),
            sum(g for g,_,_ in fm)/len(fm),
            sum(a for _,a,_ in fm)/len(fm),
            cwins/len(fm))

for _, m in historical.iterrows():
    home, away = m["home_team"], m["away_team"]
    hg, ag     = m["home_score"], m["away_score"]
    is_comp    = "friendly" not in str(m.get("tournament","")).lower()
    hw,hg2,hga,hc = form_snap(home)
    aw,ag2,aga,ac = form_snap(away)
    h_wr.append(hw); h_gf.append(hg2); h_ga.append(hga); h_cw.append(hc)
    a_wr.append(aw); a_gf.append(ag2); a_ga.append(aga); a_cw.append(ac)
    if pd.isna(hg) or pd.isna(ag): continue
    team_form[home].append((hg, ag, is_comp))
    team_form[away].append((ag, hg, is_comp))

historical["home_form_wr"]   = h_wr;  historical["home_form_gf"]   = h_gf
historical["home_form_ga"]   = h_ga;  historical["home_form_comp"] = h_cw
historical["away_form_wr"]   = a_wr;  historical["away_form_gf"]   = a_gf
historical["away_form_ga"]   = a_ga;  historical["away_form_comp"] = a_cw
print("✅ Form features built")

## 5. Head-to-Head Win Rates

In [ ]:
h2h       = defaultdict(lambda: {"w":0,"d":0,"l":0})
h2h_ha, h2h_aa = [], []

for _, m in historical.iterrows():
    home, away = m["home_team"], m["away_team"]
    hg, ag     = m["home_score"], m["away_score"]
    rh=h2h[(home,away)]; ra=h2h[(away,home)]
    th=rh["w"]+rh["d"]+rh["l"]; ta=ra["w"]+ra["d"]+ra["l"]
    h2h_ha.append(rh["w"]/th if th > 0 else 0.4)
    h2h_aa.append(ra["w"]/ta if ta > 0 else 0.4)
    if pd.isna(hg) or pd.isna(ag): continue
    if hg>ag:   rh["w"]+=1; ra["l"]+=1
    elif hg<ag: rh["l"]+=1; ra["w"]+=1
    else:       rh["d"]+=1; ra["d"]+=1

historical["h2h_home_wr"] = h2h_ha
historical["h2h_away_wr"] = h2h_aa
print(f"✅ H2H built — {len(h2h):,} unique team-pair records")

## 6. Feature Correlation Check

In [ ]:
FEATURES = [
    "home_elo_pre","away_elo_pre",
    "home_form_wr","away_form_wr",
    "home_form_gf","home_form_ga",
    "away_form_gf","away_form_ga",
    "home_form_comp","away_form_comp",
    "h2h_home_wr","h2h_away_wr",
]
corr = historical[FEATURES].corr()
plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            mask=mask, square=True, linewidths=0.4, linecolor="white",
            cbar_kws={"shrink":0.8})
plt.title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Save Features to Disk

In [ ]:
import pickle, os

# Save features CSV (for modeling notebook)
save_cols = FEATURES + ["home_score","away_score","tournament","date",
                         "home_team","away_team","neutral"]
feat_df = historical[[c for c in save_cols if c in historical.columns]].copy()
feat_df.to_csv("features/historical_features.csv", index=False)
print(f"✅ Saved: features/historical_features.csv  ({len(feat_df):,} rows)")

# Save ELO dict and team_form for prediction notebook
import json
with open("features/elo_final.json","w") as f:
    json.dump({k: round(v,2) for k,v in elo.items()}, f)
print("✅ Saved: features/elo_final.json")

# Save h2h
h2h_serializable = {str(k): v for k, v in h2h.items()}
with open("features/h2h.json","w") as f:
    json.dump(h2h_serializable, f)
print("✅ Saved: features/h2h.json")

# Save wc26_fix
wc26_fix.to_csv("features/wc26_fixtures.csv", index=False)
print(f"✅ Saved: features/wc26_fixtures.csv  ({len(wc26_fix)} fixtures)")

---
## ✅ Feature Engineering Complete

**16 features built, 0 data leakage** — all computed from data strictly before each match.

**Outputs saved to `features/`:**
- `historical_features.csv` — full feature matrix for modeling
- `elo_final.json` — final ELO ratings for all teams
- `h2h.json` — head-to-head records
- `wc26_fixtures.csv` — WC 2026 group fixtures

**→ Next notebook:** `03_modeling.ipynb`
